# Connection - cost jump
Connection only: the analysis notebooks in this folder copy the connection cell below unchanged as their first code cell.
- Set the environment variables `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_ROLE`, `SNOWFLAKE_WAREHOUSE` and either `SNOWFLAKE_PRIVATE_KEY_FILE` (key pair) or `SNOWFLAKE_AUTHENTICATOR` (default `externalbrowser`) before starting Jupyter.
- `TIMEZONE = 'UTC'` and the query tag are set as session parameters, so no `ALTER SESSION` is needed.
- Snowflake Notebook instead of Jupyter: replace the connection cell with `session = get_active_session()` and `q = lambda sql, params=None: session.sql(sql, params=params).to_pandas()`.

In [ ]:
# Connection only. Every value comes from an environment variable; nothing secret is stored here.
import os

import pandas as pd
import snowflake.connector

key_file = os.environ.get("SNOWFLAKE_PRIVATE_KEY_FILE")
auth = ({"authenticator": "SNOWFLAKE_JWT", "private_key_file": key_file} if key_file
        else {"authenticator": os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser")})
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    role=os.environ["SNOWFLAKE_ROLE"],            # a read-only role
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],  # X-Small is enough for system views
    session_parameters={"TIMEZONE": "UTC", "QUERY_TAG": "analysis:cost-jump"},
    **auth,
)


def q(sql, params=None):
    """Run one read-only statement and return a DataFrame (SHOW output too)."""
    cur = conn.cursor()
    try:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    finally:
        cur.close()

### Connection test
- **What:** shows the account, role, warehouse and version this notebook runs with.
- **Reads:** context functions only; no view.
- **Look for:** the role is the read-only analysis role (never an admin role) and the warehouse is X-Small.

In [ ]:
q("""SELECT CURRENT_ACCOUNT() AS account, CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse, CURRENT_VERSION() AS version""")

   ACCOUNT        ROLE    WAREHOUSE  VERSION
0  XY12345  ANALYST_RO  ANALYSIS_XS   9.36.2